# tools — operator and diagnostic commands

In [ ]:
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
CONFIG = ROOT / 'examples/e2e/config.json'
RUN_S3 = os.getenv('DTRACK_RUN_S3', 's3://dtrack-e2e-work/dtrack/v1/runs')
RUN_ID = os.getenv('DTRACK_RUN_ID', 'nb-distributed')
PAIR = 'orders'
EXECUTE_READ_ONLY = True  # doctor uses the generated local Left files; Right is only described.

subprocess.run([sys.executable, str(ROOT / 'examples/e2e/make_synthetic.py')], cwd=ROOT, check=True, capture_output=True)

In [ ]:
def dtrack(*args, execute=False):
    command = [sys.executable, '-m', 'dtrack', *map(str, args)]
    print('$', shlex.join(command))
    return subprocess.run(command, cwd=ROOT, text=True, check=execute) if execute else command

# doctor reaches every Left source and describes, but does not query, each Right relation.
doctor = dtrack('tools', 'doctor', '--config', CONFIG, execute=EXECUTE_READ_ONLY)
doctor

In [ ]:
# Materialize against a scratch copy: verified Parquet first, config switch second, CSV deletion last.
SCRATCH = ROOT / '.dtrack-examples/materialize'
if SCRATCH.exists(): shutil.rmtree(SCRATCH)
SCRATCH.mkdir(parents=True)
source_config = json.loads(CONFIG.read_text())
source_csv = ROOT / source_config['pairs'][PAIR]['left']['path']
scratch_csv = SCRATCH / source_csv.name
shutil.copy2(source_csv, scratch_csv)
source_config['pairs'] = {PAIR: source_config['pairs'][PAIR]}
source_config['pairs'][PAIR]['left']['path'] = str(scratch_csv)
scratch_config = SCRATCH / 'config.json'
scratch_config.write_text(json.dumps(source_config, indent=2))
materialize = dtrack('tools', 'materialize', '--config', scratch_config, '--pair', PAIR, '--delete-source', execute=True)
materialized = json.loads(scratch_config.read_text())['pairs'][PAIR]['left']
assert not scratch_csv.exists()
assert materialized['format'] == 'parquet' and materialized['date_type'] == 'date'
assert materialized['sample_hash'] == {'column': 'dtrack_hash', 'version': 1, 'keys': ['ORDER_ID'], 'families': ['numeric']}
materialized_root = Path(materialized['path'].split('/**/')[0])
assert materialized_root.joinpath('_dtrack_materialization.json').is_file()
materialized

In [ ]:
# Recovery/report commands are scoped by the same runs root and run-id. Rollback is dry-run without --yes.
commands = {
    'fetch_report': dtrack('tools', 'fetch-report', '--run-s3', RUN_S3, '--run-id', RUN_ID, '--out', ROOT / '.dtrack-reports' / RUN_ID),
    'rollback_plan': dtrack('tools', 'rollback', '--run-s3', RUN_S3, '--run-id', RUN_ID, '--pair', PAIR, '--stage', 'col'),
    'debug_local': dtrack('tools', 'debug', '--', 'local', 'preflight', '--config', CONFIG, '--pair', PAIR),
    'dbx_login': dtrack('tools', 'dbx', 'login'),
    'dbx_sql': dtrack('tools', 'dbx', 'sql', '--help'),
    'blog': dtrack('tools', 'blog', '--help'),
}
commands